In [0]:
from pyspark.sql import functions as F, types as T
rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])
df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders    = spark.createDataFrame(rows_orders,    schema_orders)

display(df_customers)
display(df_orders)

customer_id,name,country,vip
1,Asha,IN,true
2,Bob,US,false
3,Chen,CN,true
4,Diana,US,null
null,Ghost,UK,false


order_id,customer_id,amount,country
101,1,120.0,IN
102,1,80.0,IN
103,2,50.0,US
104,5,30.0,DE
105,3,200.0,CN
106,null,15.0,UK
107,3,40.0,CN
108,2,75.0,US


In [0]:
df_inner = df_orders.join(df_customers, "customer_id")
display(df_inner)

customer_id,order_id,amount,country,name,country,vip
1,101,120.0,IN,Asha,IN,true
1,102,80.0,IN,Asha,IN,true
2,103,50.0,US,Bob,US,false
3,105,200.0,CN,Chen,CN,true
3,107,40.0,CN,Chen,CN,true
2,108,75.0,US,Bob,US,false


## we can also specify the type of join as following

In [0]:
df_inner1 = df_orders.join(df_customers, on="customer_id", how="left")
display(df_inner1)

customer_id,order_id,amount,country
1,101,120.0,IN
1,102,80.0,IN
2,103,50.0,US
3,105,200.0,CN
3,107,40.0,CN
2,108,75.0,US


- **LEFT SEMI** → finds rows in the left table that have a match in the right table, and returns only left table columns
- **LEFT ANTI** → finds rows in the left table that do NOT have a match in the right table, and returns only left table columns
- **RIGHT SEMI** and **RIGHT ANTI** doesnot exist in practical 

In [0]:
display(df_orders.join(df_customers, on="customer_id", how="left_semi"))

customer_id,order_id,amount,country
1,101,120.0,IN
1,102,80.0,IN
2,103,50.0,US
3,105,200.0,CN
3,107,40.0,CN
2,108,75.0,US
